#1. Install Packages

In [ ]:
# ==========================================================
# Install Required Packages
# ==========================================================

!pip install -q chromadb
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q langchain-chroma
!pip install -q openpyxl

#2. Import Libraries

In [ ]:
# ==========================================================
# Import Libraries
# ==========================================================

import os
import time
import pandas as pd

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.documents import Document

from langchain_chroma import Chroma

#3. Project Paths

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/RAG_ICH/data/"))

In [ ]:
# ==========================================================
# Project Paths
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/RAG_ICH"

CHUNK_DIR = os.path.join(PROJECT_DIR, "data/chunks")

EVAL_FILE = os.path.join(
    PROJECT_DIR,
    "data/evaluation/evaluation_questions.xlsx"
)

VECTOR_DIR = os.path.join(
    PROJECT_DIR,
    "vectordb"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results/chunking"
)

os.makedirs(VECTOR_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/RAG_ICH"

print(f"Does PROJECT_DIR exist? {os.path.exists(PROJECT_DIR)}\n")

for root, dirs, files in os.walk(PROJECT_DIR):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = '  ' * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

# Load Evaluation Questions

In [ ]:
evaluation_df = pd.read_excel(EVAL_FILE)

print(f"Number of evaluation questions: {len(evaluation_df)}")

display(evaluation_df.head())

#5. Select Chunking Strategy (Reproducible)

In [ ]:
# ==========================================================
# Select Chunking Strategy
# ==========================================================

CHUNK_FILE = "semantic_chunks.csv"

# Change to:
# "fixed_chunks.csv"
# "recursive_chunks.csv"
# "semantic_chunks.csv"

chunk_path = os.path.join(CHUNK_DIR, CHUNK_FILE)

chunk_df = pd.read_csv(chunk_path)

print(chunk_df.head())

#6. Create Embedding Model

In [ ]:
# ==========================================================
# Create Embedding Model
# ==========================================================

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda"},      # Change to "cpu" if GPU is unavailable
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded successfully.")

#7. Convert Chunks into Langchain Document

In [ ]:
# ==========================================================
# Convert Chunks into LangChain Documents
# ==========================================================

from langchain_core.documents import Document

documents = []

for _, row in chunk_df.iterrows():

    doc = Document(
        page_content=row["text"],
        metadata={
            "chunk_id": row["chunk_id"],
            "chunking_method": CHUNK_FILE.replace(".csv", "")
        }
    )

    documents.append(doc)

print(f"Total documents created: {len(documents)}")
print(documents[0])

In [ ]:
# ==========================================================
# Inspect First Document
# ==========================================================

print("="*80)
print("Chunk ID:", documents[0].metadata["chunk_id"])
print("Chunking Method:", documents[0].metadata["chunking_method"])
print("="*80)
print(documents[0].page_content[:800])

#8. Create ChromaDB

In [ ]:
# ==========================================================
# Create Chroma Vector Database
# ==========================================================

import os
from langchain_chroma import Chroma

# Collection name based on the chunking strategy
collection_name = CHUNK_FILE.replace(".csv", "")

# Save each strategy in its own folder
persist_directory = os.path.join(
    VECTOR_DIR,
    collection_name
)

print(f"Collection Name : {collection_name}")
print(f"Database Folder : {persist_directory}")

# Create the Chroma database
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name=collection_name,
    persist_directory=persist_directory
)

print("\n✅ Chroma Vector Database created successfully!")

#9. Verify db

In [ ]:
# ==========================================================
# Verify Chroma Database
# ==========================================================

collection = vector_db._collection

print(f"Collection Name : {collection.name}")
print(f"Number of vectors : {collection.count()}")

In [ ]:
# ==========================================================
# Inspect Stored Metadata
# ==========================================================

sample = vector_db.get(limit=1)

print(sample.keys())
print()

print("Metadata:")
print(sample["metadatas"][0])

print("\nDocument:")
print(sample["documents"][0][:500])

#Create Retriever

In [ ]:
# ==========================================================
# Create Retriever
# ==========================================================

retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

print("Retriever created successfully.")

#Test Retriever

In [ ]:
display(evaluation_df.head())

In [ ]:
# ==========================================================
# Test Retrieval
# ==========================================================

query = evaluation_df.loc[
    evaluation_df["question_id"]=="Q002",
    "clinical_question"
].values[0]

print(query)

In [ ]:
results = retriever.invoke(query)

print(f"Retrieved {len(results)} chunks\n")

for i, doc in enumerate(results):

    print("="*80)
    print(f"Rank {i+1}")
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("-"*80)
    print(doc.page_content[:600])
    print()

In [ ]:
# ==========================================================
# Retrieve Top-5 Chunks for All Evaluation Questions
# ==========================================================

retrieval_results = []

for _, row in evaluation_df.iterrows():

    question_id = row["question_id"]
    question = row["clinical_question"]

    docs = retriever.invoke(question)

    for rank, doc in enumerate(docs, start=1):

        retrieval_results.append({

        "question_id": question_id,
        "domain": row["domain"],
        "subdomain": row["subdomain"],
        "clinical_question": question,
        "reference_answer": row["reference_answer_draft"],
        "gold_evidence": row["gold_evidence_location"],

        "rank": rank,
        "chunk_id": doc.metadata["chunk_id"],
        "retrieved_text": doc.page_content

    })

retrieval_df = pd.DataFrame(retrieval_results)

print(f"Total retrieved rows: {len(retrieval_df)}")

display(retrieval_df.head())

In [ ]:
# ==========================================================
# Save Retrieval Results
# ==========================================================

# # Automatically detect the current chunking strategy
# strategy = CHUNK_FILE.replace(".csv", "")

# # Output filename
# output_file = os.path.join(
#     RESULT_DIR,
#     f"{strategy}_retrieval_results.csv"
# )

# # Save CSV
# retrieval_df.to_csv(output_file, index=False)

# print("="*60)
# print("Retrieval results saved successfully!")
# print(f"Strategy : {strategy}")
# print(f"Rows     : {len(retrieval_df)}")
# print(f"Location : {output_file}")
# print("="*60)